In [5]:
import numpy as np
import pandas as pd

# CSV files exported from Earth Engine (same folder as this notebook)
FILES = {
    'era5land': '/Users/sophiethomas/Desktop/MUDE /MDP/Climatology/Sisal_ERA5Land_daily_all.csv',
    'era5':     '/Users/sophiethomas/Desktop/MUDE /MDP/Climatology/Sisal_ERA5_daily_all.csv',
    'hycom':    '/Users/sophiethomas/Desktop/MUDE /MDP/Climatology/Sisal_HYCOM_daily_all.csv',
}

# kind:    'linear'   ordinary variable (averaged)
#          'flux'     daily amount, e.g. mm/day (summed to monthly totals)
#          'circular' direction in degrees (averaged as a vector)
# monthly: optional override of the monthly aggregation ('mean', 'sum', 'max')
# Units are already converted in the Earth Engine export.
VARIABLES = {
    # ERA5-Land, two inland cells (choose CELL = 'c1' or 'c2')
    't2m_c':       dict(src='era5land', label='2 m air temperature (daily mean)', unit='°C',   kind='linear'),
    't2m_min_c':   dict(src='era5land', label='2 m air temperature (daily min)',  unit='°C',   kind='linear'),
    't2m_max_c':   dict(src='era5land', label='2 m air temperature (daily max)',  unit='°C',   kind='linear'),
    'skin_t_c':    dict(src='era5land', label='Skin temperature',                  unit='°C',   kind='linear'),
    'precip_mm':   dict(src='era5land', label='Precipitation',                     unit='mm',   kind='flux'),
    'evap_mm':     dict(src='era5land', label='Actual evaporation',                unit='mm',   kind='flux'),
    'pot_evap_mm': dict(src='era5land', label='Potential evaporation',             unit='mm',   kind='flux'),
    'evap_gap_mm': dict(src='era5land', label='Evaporation gap (potential - actual)', unit='mm', kind='flux'),
    # ERA5, sea cell
    'wind_u_ms':       dict(src='era5', label='10 m wind, eastward component',  unit='m/s',  kind='linear'),
    'wind_v_ms':       dict(src='era5', label='10 m wind, northward component', unit='m/s',  kind='linear'),
    'wind_speed_ms':   dict(src='era5', label='10 m wind speed',                unit='m/s',  kind='linear'),
    'sst_c':           dict(src='era5', label='Sea surface temperature (ERA5)', unit='°C',   kind='linear'),
    'hs_mean_m':       dict(src='era5', label='Significant wave height (mean)', unit='m',    kind='linear'),
    'hs_max_m':        dict(src='era5', label='Significant wave height (max)',  unit='m',    kind='linear', monthly='max'),
    'tp_mean_s':       dict(src='era5', label='Peak wave period',               unit='s',    kind='linear'),
    'wave_dir_deg':    dict(src='era5', label='Mean wave direction (from)',     unit='°',    kind='circular'),
    'wave_power_kw_m': dict(src='era5', label='Wave power',                     unit='kW/m', kind='linear'),
    # HYCOM, one pixel
    'temp_surface_c':      dict(src='hycom', label='Sea surface temperature (HYCOM)', unit='°C',  kind='linear'),
    'temp_bottom_c':       dict(src='hycom', label='Bottom temperature',              unit='°C',  kind='linear'),
    'salinity_surface_psu':dict(src='hycom', label='Surface salinity',                unit='PSU', kind='linear'),
    'salinity_bottom_psu': dict(src='hycom', label='Bottom salinity',                 unit='PSU', kind='linear'),
}


def load_series(var, cell='c1'):
    """Daily series on a complete daily index (missing days are NaN)."""
    info = VARIABLES[var]
    path = FILES[info['src']]
    df = pd.read_csv(path, parse_dates=['date'])
    col = f'{var}_{cell}' if info['src'] == 'era5land' else var
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found in {path}. Available: {list(df.columns)}")

    if info['src'] == 'hycom' and 'bottom_level_m' in df.columns:
        levels = df['bottom_level_m'].dropna().unique()
        print(f'HYCOM bottom level(s) used: {sorted(levels)} m'
              + ('  <-- should be a single value' if len(levels) > 1 else ''))

    s = df.set_index('date')[col].sort_index()
    s = s[~s.index.duplicated()]
    s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq='D'))
    s.name = var

    n_years = (s.index[-1] - s.index[0]).days / 365.25
    print(f'{info["label"]} [{col}]: {s.index[0].date()} to {s.index[-1].date()}, '
          f'{n_years:.1f} years, {s.isna().sum()} missing days')
    if n_years < 20:
        print(f'WARNING: only {n_years:.0f} years of data. Trends and percentiles are '
              'not robust; use this run to test the pipeline, not to report results.')
    return s, info


In [6]:
from scipy import stats

VAR  = 'precip_mm'      # any key from VARIABLES except 'wave_dir_deg'
CELL = 'c1'         # ERA5-Land only: rerun with 'c2' to cross-check
WINDOW, SMOOTH = 11, 31    # days: percentile window and smoothing
DETREND = False             # None = automatic (only if >= 20 years of data)

s, info = load_series(VAR, CELL)
if info['kind'] == 'circular':
    raise ValueError('Directions cannot be detrended or put into percentiles. '
                     'Use Climatology_plots.ipynb for wave direction.')

TAG   = CELL if info['src'] == 'era5land' else info['src']
UNIT  = info['unit'] + ('/day' if info['kind'] == 'flux' else '')
valid = s.dropna()
YEAR0, YEAR1 = s.index[0].year, s.index[-1].year
N_YEARS = (s.index[-1] - s.index[0]).days / 365.25
if DETREND is None:
    DETREND = N_YEARS >= 20

# --- 1. linear trend on daily anomalies (seasonal cycle removed) -----------
# --- day-of-year index, leap years collapsed onto a 365-day calendar -------
def doy365(idx):
    doy = idx.dayofyear.values.copy()
    doy[idx.is_leap_year & (doy > 59)] -= 1      # 29 Feb shares day 59
    return doy


doy_all  = doy365(s.index)
raw_clim = pd.Series(s.values, index=doy_all).groupby(level=0).mean()
anom     = s - raw_clim.reindex(doy_all).values

t_all = (s.index - s.index[0]).days.values / 365.25
t     = t_all[anom.notna().values]
slope, icpt = np.polyfit(t, anom.dropna().values, 1)
print(f'daily-fit trend: {slope*10:.3f} {UNIT} per decade')

# --- 2. trend with uncertainty, on annual means (years >= 90% complete) ----
g   = s.resample('YE')
ann = g.mean()[g.count() >= 329]
if len(ann) >= 3:
    res = stats.linregress(ann.index.year, ann.values)
    print(f'annual-fit trend: {res.slope*10:.3f} '
          f'± {res.stderr*10*1.96:.3f} {UNIT} per decade (95%)')
    print('Mann-Kendall:', stats.kendalltau(ann.index.year, ann.values))
else:
    res = None
    print('Too few complete years for an annual trend.')

# --- 3. detrend (keeps the mean level) --------------------------------------
x = s - slope * (t_all - t.mean()) if DETREND else s.copy()
print('detrended before climatology:', DETREND)

# --- 4. day-of-year index, leap years collapsed onto a 365-day calendar ----

doy = doy365(x.index)
r   = pd.Series(x.values, index=doy).dropna()

# --- 5. WINDOW-day climatology, wrapping at the year boundary --------------
half = WINDOW // 2
clim = {}
for d in range(1, 366):
    w = [(d + k - 1) % 365 + 1 for k in range(-half, half + 1)]
    v = r[r.index.isin(w)].values
    clim[d] = (v.mean(), np.percentile(v, 10), np.percentile(v, 90))
clim = pd.DataFrame(clim, index=['mean', 'p10', 'p90']).T
print(f'values per day-of-year window: ~{len(r) * WINDOW // 365}')

# --- 5b. precipitation: wet-day frequency and intensity --------------------
WET = 1.0   # mm, ETCCDI wet-day threshold
if VAR == 'precip_mm':
    wet = {}
    for d in range(1, 366):
        w  = [(d + k - 1) % 365 + 1 for k in range(-half, half + 1)]
        v  = r[r.index.isin(w)].values
        vw = v[v >= WET]
        if len(vw) >= 20:
            wet[d] = ((v >= WET).mean(), vw.mean(), np.percentile(vw, 90))
        else:
            wet[d] = ((v >= WET).mean(), np.nan, np.nan)
    wet = pd.DataFrame(wet, index=['wet_freq', 'wet_mean', 'wet_p90']).T
    wet = (pd.concat([wet] * 3)
             .rolling(SMOOTH, center=True, min_periods=SMOOTH // 2)
             .mean().iloc[365:730])
    wet.index = range(1, 366)
    print(wet.describe())

# --- 6. circular SMOOTH-day smoothing ----------------------------------------
clim = pd.concat([clim] * 3).rolling(SMOOTH, center=True).mean().iloc[365:730]
clim.index = range(1, 366)

print(clim.describe())
#clim.to_csv(f'climatology_{VAR}_{TAG}.csv', index_label='doy')


Precipitation [precip_mm_c1]: 1950-01-02 to 2025-12-31, 76.0 years, 0 missing days
daily-fit trend: -0.023 mm/day per decade
annual-fit trend: -0.023 ± 0.038 mm/day per decade (95%)
Mann-Kendall: SignificanceResult(statistic=np.float64(-0.12000000000000001), pvalue=np.float64(0.12507332582449004))
detrended before climatology: False
values per day-of-year window: ~836
         wet_freq    wet_mean     wet_p90
count  365.000000  365.000000  365.000000
mean     0.323760    4.338700    9.020699
std      0.189758    0.635103    1.407584
min      0.094808    3.519362    7.133247
25%      0.152030    3.874883    7.938681
50%      0.258335    4.081761    8.515174
75%      0.501158    4.803888    9.974666
max      0.661831    5.678246   12.676921
             mean         p10         p90
count  365.000000  365.000000  365.000000
mean     1.622546    0.033143    4.079751
std      1.051901    0.052343    2.654296
min      0.451251    0.000432    0.906624
25%      0.725335    0.000678    1.730415

In [7]:
import numpy as np
import pandas as pd

BASE = 'https://www.ncei.noaa.gov/pub/data/ghcn/daily/'
inv = pd.read_fwf(BASE + 'ghcnd-inventory.txt', header=None,
                  colspecs=[(0, 11), (12, 20), (21, 30), (31, 35), (36, 40), (41, 45)],
                  names=['id', 'lat', 'lon', 'elem', 'first', 'last'])
st  = pd.read_fwf(BASE + 'ghcnd-stations.txt', header=None,
                  colspecs=[(0, 11), (41, 71)], names=['id', 'name'])

p = inv[(inv.elem == 'PRCP') & inv.id.str.startswith('MX')].merge(st, on='id')


SITE = (21.165, -90.03)   # Sisal; replace with your mangrove coordinates if you have them
la1, lo1 = np.radians(SITE)
la2, lo2 = np.radians(p.lat), np.radians(p.lon)
p['km'] = 6371 * 2 * np.arcsin(np.sqrt(np.sin((la2 - la1) / 2)**2
                                       + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2)**2))

print(p.nsmallest(10, 'km')[['id', 'name', 'first', 'last', 'km']].to_string(index=False))
#uncomment the line below to check for the temperature 
#print(inv[inv.id == 'MXN00031029'][['elem', 'first', 'last']].to_string(index=False))

         id                      name  first  last        km
MXN00031029                     SISAL   1957  2007  2.164295
MXN00031051                   HUNUCMA   1974  1993 23.642105
MXN00031016                   KINCHIL   1966  2008 30.614089
MXM00076593                  PROGRESO   1988  2024 41.526531
MXN00031078 F.I.U.A.D.Y. (PERIFERICO)   1982  2000 43.681676
MXN00031052                      UMAN   1974  1991 44.101008
MXM00076644       AEROP.INTERNACIONAL   1973  2025 44.318380
MXN00031007          CHICXULUB PUERTO   1963  2007 46.471276
MXN00031044             MERIDA CENTRO   1961  2000 48.221991
MXN00031079           MERIDA SANTIAGO   1982  2013 50.560794


In [8]:
URL = 'https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/MXN00031029.csv'
g = pd.read_csv(URL, parse_dates=['DATE'], usecols=['DATE', 'PRCP', 'PRCP_ATTRIBUTES'])

qflag = g['PRCP_ATTRIBUTES'].fillna(',,,').str.split(',').str[1]
g.loc[qflag != '', 'PRCP'] = np.nan          # drop QC-flagged values

obs = g.set_index('DATE')['PRCP'] / 10       # tenths of mm -> mm
obs = obs.reindex(pd.date_range(obs.index.min(), obs.index.max(), freq='D'))

per_year = obs.groupby(obs.index.year).agg(['count', 'sum'])
print(per_year.to_string())
print('years with >= 329 days:', (per_year['count'] >= 329).sum())

      count     sum
1957     28    13.0
1958    365   987.4
1959    365   744.8
1960    366   636.2
1961    365   484.9
1962    365   262.0
1963    365   418.1
1964    364   332.2
1965    365   855.4
1966    364   813.3
1967    365   247.3
1968    366   355.7
1969    365   708.4
1970    365   506.6
1971    365   305.1
1972    366   614.7
1973    365   574.3
1974    365  1086.2
1975    365   535.7
1976    366   691.3
1977    365   531.9
1978    365   667.7
1979    365   686.8
1980    335   614.3
1981    365   360.6
1982    331   275.9
1983    334   477.0
1984    366   540.8
1985    334   568.7
1986    365   230.1
1987    215   362.6
1988    243   525.8
1989    272   300.9
1990    334   119.1
1991    365    50.2
1992    366   383.4
1993    364   330.1
1994    365   275.2
1995    365   443.3
1996    366   302.2
1997    365   516.0
1998    184   518.3
1999    334   688.4
2000    366   690.6
2001    365   459.5
2002    365   780.7
2003    365   592.8
2004    366   682.2
2005    365   532.3


In [9]:
era = s.groupby(s.index.year).sum()
cmp = pd.DataFrame({'n': per_year['count'], 'station': per_year['sum'], 'era5': era})
cmp = cmp[cmp['n'] >= 329].dropna()
cmp['ratio'] = cmp['station'] / cmp['era5']
print(cmp.round(2).to_string())

          n  station    era5  ratio
1958  365.0    987.4  685.66   1.44
1959  365.0    744.8  847.52   0.88
1960  366.0    636.2  529.05   1.20
1961  365.0    484.9  716.45   0.68
1962  365.0    262.0  393.68   0.67
1963  365.0    418.1  574.44   0.73
1964  364.0    332.2  576.25   0.58
1965  365.0    855.4  569.64   1.50
1966  364.0    813.3  592.34   1.37
1967  365.0    247.3  609.04   0.41
1968  366.0    355.7  471.07   0.76
1969  365.0    708.4  756.69   0.94
1970  365.0    506.6  474.50   1.07
1971  365.0    305.1  477.39   0.64
1972  366.0    614.7  607.32   1.01
1973  365.0    574.3  755.65   0.76
1974  365.0   1086.2  810.22   1.34
1975  365.0    535.7  469.67   1.14
1976  366.0    691.3  607.38   1.14
1977  365.0    531.9  536.08   0.99
1978  365.0    667.7  587.07   1.14
1979  365.0    686.8  604.42   1.14
1980  335.0    614.3  705.35   0.87
1981  365.0    360.6  626.71   0.58
1982  331.0    275.9  639.74   0.43
1983  334.0    477.0  846.16   0.56
1984  366.0    540.8  835.92

In [10]:
bad = [1990, 1991]
o = obs[~obs.index.year.isin(bad)]
both = pd.DataFrame({'station': o, 'era5': s.reindex(o.index)}).dropna()

def wet_stats(df):
    out = {}
    for c in ['station', 'era5']:
        v = df[c]
        w = v[v >= 1.0]
        out[c] = {'wet_days_per_yr': (v >= 1.0).mean() * 365.25,
                  'wet_mean_mm': w.mean(),
                  'wet_p90_mm': w.quantile(0.90),
                  'wet_p99_mm': w.quantile(0.99)}
    return pd.DataFrame(out).round(1)

print('drop 1990-91 only');  print(wet_stats(both))
susp = both.index.year.isin(range(1981, 1997))
print('\nalso drop 1981-96'); print(wet_stats(both[~susp]))

drop 1990-91 only
                 station   era5
wet_days_per_yr     42.7  120.8
wet_mean_mm         13.1    4.7
wet_p90_mm          30.0    9.8
wet_p99_mm          84.3   24.8

also drop 1981-96
                 station   era5
wet_days_per_yr     45.2  118.3
wet_mean_mm         13.6    4.6
wet_p90_mm          31.2    9.6
wet_p99_mm          91.5   23.3


In [11]:
m = both.groupby(both.index.month).mean() * 30.44     # mean daily rate -> mm/month
m['ratio'] = m['era5'] / m['station']
print(m.round(1).to_string())
print('\nannual mean (mm):', (both.mean() * 365.25).round(0).to_dict())

c = cmp.drop([1990, 1991], errors='ignore')
print('year-to-year correlation r:', round(c['station'].corr(c['era5']), 2))

    station   era5  ratio
1      27.5   24.7    0.9
2      20.2   22.6    1.1
3       9.8   12.8    1.3
4       9.6   15.5    1.6
5      31.7   36.9    1.2
6      81.4   88.9    1.1
7      75.0   62.6    0.8
8      65.5   74.5    1.1
9     105.8  129.2    1.2
10     71.2   77.7    1.1
11     29.6   37.3    1.3
12     35.4   27.5    0.8

annual mean (mm): {'station': 564.0, 'era5': 611.0}
year-to-year correlation r: 0.35


## From here on down we're checking temperature

In [12]:
g = pd.read_csv(URL, parse_dates=['DATE'])
idx = pd.date_range(g['DATE'].min(), g['DATE'].max(), freq='D')

tobs = {}
for el in ['TMAX', 'TMIN']:
    q = g[f'{el}_ATTRIBUTES'].fillna(',,,').str.split(',').str[1]
    v = g[el].where(q == '') / 10                 # tenths of °C -> °C, QC-flagged removed
    tobs[el] = pd.Series(v.values, index=g['DATE']).reindex(idx)
tobs = pd.DataFrame(tobs)

print(tobs.groupby(tobs.index.year).agg(['count', 'mean']).round(1).to_string())

      TMAX        TMIN      
     count  mean count  mean
1957    28  26.4    28  18.5
1958   365  30.3   365  20.1
1959   365  30.7   342  19.8
1960   364  29.4   316  16.8
1961   365  28.7   365  19.6
1962   364  28.5   365  18.3
1963   365  30.4   361  18.8
1964   363  31.3   363  20.3
1965   365  31.8   365  20.6
1966   364  30.6   364  20.5
1967   365  30.8   365  20.7
1968   366  30.3   366  20.0
1969   361  30.7   365  20.7
1970   365  30.7   365  20.3
1971   365  31.5   365  20.3
1972   365  30.8   366  21.1
1973   365  30.8   365  21.0
1974   365  30.6   365  20.6
1975   365  31.8   365  20.7
1976   366  30.4   366  20.3
1977   365  31.4   365  20.5
1978   365  31.0   365  20.4
1979   365  30.9   365  20.8
1980   335  31.8   335  20.1
1981   243  32.7   243  19.2
1982   303  33.3   303  21.5
1983   333  33.4   333  20.6
1984   366  30.8   366  20.8
1985   334  31.8   334  20.5
1986   365  32.5   365  20.8
1987   215  33.7   215  21.5
1988   243  34.4   243  20.6
1989   271  32

In [13]:
emax, _ = load_series('t2m_max_c', CELL)
emin, _ = load_series('t2m_min_c', CELL)

per = slice('1964-01-01', '1980-12-31')
ct = pd.DataFrame({'st_max':  tobs['TMAX'][per], 'era_max': emax[per],
                   'st_min':  tobs['TMIN'][per], 'era_min': emin[per]}).dropna()

mt = ct.groupby(ct.index.month).mean()
mt['bias_max'] = mt['era_max'] - mt['st_max']
mt['bias_min'] = mt['era_min'] - mt['st_min']
print(mt.round(1).to_string())
print('\nmean bias  max:', round(mt['bias_max'].mean(), 1),
      '  min:', round(mt['bias_min'].mean(), 1))

2 m air temperature (daily max) [t2m_max_c_c1]: 1950-01-02 to 2025-12-31, 76.0 years, 0 missing days
2 m air temperature (daily min) [t2m_min_c_c1]: 1950-01-02 to 2025-12-31, 76.0 years, 0 missing days
    st_max  era_max  st_min  era_min  bias_max  bias_min
1     28.1     26.9    18.2     20.2      -1.2       2.1
2     28.7     27.4    17.9     20.1      -1.3       2.3
3     31.6     30.5    19.4     21.9      -1.1       2.5
4     33.5     32.4    20.7     23.4      -1.1       2.7
5     34.2     32.7    21.8     24.7      -1.4       2.9
6     32.7     31.7    22.3     24.8      -1.0       2.6
7     32.0     31.8    22.0     24.6      -0.2       2.7
8     32.0     31.8    22.0     24.7      -0.2       2.7
9     31.6     30.7    22.2     24.5      -0.9       2.3
10    30.4     28.9    21.6     23.6      -1.5       2.0
11    28.9     27.8    20.0     22.1      -1.2       2.1
12    28.3     26.9    18.4     20.8      -1.4       2.4

mean bias  max: -1.0   min: 2.4


In [ ]:
# #In the main cell, set:
# VAR     = 't2m_max_c'
# DETREND = False      # decide after seeing the trend